# TCGA-BRCA Blueprint Ambiguity Resolution Review

This notebook is review-only. It loads the latest saved blueprint ambiguity-resolution outputs from disk, checks the run-level validation state, and writes review tables for human audit.


## Load the latest saved ambiguity-resolution run


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'cohort'
    / 'tcga_brca_blueprint_ambiguity_resolution_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest ambiguity-resolution pointer not found: {latest_pointer_path}. Run the ambiguity-resolution script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
inventory_path = repo_root / latest_pointer['ambiguity_resolution_inventory_tsv']
priority_path = repo_root / latest_pointer['ambiguity_resolution_priority_tsv']
actions_path = repo_root / latest_pointer['ambiguity_resolution_actions_tsv']
summary_path = repo_root / latest_pointer['ambiguity_resolution_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


## Load saved ambiguity-resolution artifacts


In [ ]:
inventory_df = pd.read_csv(inventory_path, sep='\t')
priority_df = pd.read_csv(priority_path, sep='\t')
actions_df = pd.read_csv(actions_path, sep='\t')
summary_df = pd.read_csv(summary_path, sep='\t')
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

print(f"Ambiguity-resolution run ID: {latest_pointer['ambiguity_resolution_run_id']}")
print(f"Blueprint run ID: {latest_pointer['blueprint_run_id']}")
print(f"Inventory TSV: {inventory_path}")
print(f"Priority TSV: {priority_path}")
print(f"Actions TSV: {actions_path}")
print(f"Summary TSV: {summary_path}")
print(f"Run log: {run_log_path}")

validation_df = pd.DataFrame([run_log['validation']])
display(validation_df)


## Review counts and highest-priority ambiguities


In [ ]:
inventory_sorted_df = inventory_df.sort_values(
    ['blocker_class', 'resolution_priority', 'ambiguity_id'],
    ascending=[True, True, True],
).reset_index(drop=True)
blocking_counts_df = (
    inventory_df.groupby('blocker_class', dropna=False)
    .size()
    .reset_index(name='ambiguity_count')
    .sort_values(['ambiguity_count', 'blocker_class'], ascending=[False, True])
    .reset_index(drop=True)
)
highest_priority_df = priority_df.sort_values('priority_rank').reset_index(drop=True)
provisional_rule_df = inventory_sorted_df.loc[
    inventory_sorted_df['recommended_action_type'] == 'provisional_rule'
].reset_index(drop=True)
manual_review_df = inventory_sorted_df.loc[
    inventory_sorted_df['recommended_action_type'] == 'manual_review'
].reset_index(drop=True)
later_xml_df = inventory_sorted_df.loc[
    inventory_sorted_df['recommended_action_type'] == 'later_xml_validation'
].reset_index(drop=True)
readiness_df = summary_df.loc[
    summary_df['summary_metric'] == 'minimal_build_readiness_signal'
].reset_index(drop=True)

display(blocking_counts_df)
display(readiness_df)
display(highest_priority_df)
display(provisional_rule_df)
display(manual_review_df)
display(later_xml_df)


## Save review tables


In [ ]:
blocking_review_df = inventory_sorted_df.loc[
    inventory_sorted_df['blocker_class'] == 'blocking_now'
].reset_index(drop=True)
provisional_review_df = provisional_rule_df.copy()
manual_review_candidates_df = manual_review_df.copy()
later_xml_review_df = later_xml_df.copy()
actions_review_df = (
    priority_df.merge(
        actions_df,
        on=['ambiguity_resolution_run_id', 'action_id', 'action_title', 'needed_before_minimal_build'],
        how='left',
        validate='one_to_one',
    )
    .sort_values('priority_rank')
    .reset_index(drop=True)
)
summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)

inventory_review_path = results_root / '54_blueprint_ambiguity_resolution_inventory.tsv'
blocking_review_path = results_root / '55_blueprint_blocking_ambiguities.tsv'
provisional_review_path = results_root / '56_blueprint_provisional_rule_candidates.tsv'
manual_review_path = results_root / '57_blueprint_manual_review_ambiguities.tsv'
later_xml_review_path = results_root / '58_blueprint_later_xml_validation_candidates.tsv'
actions_review_path = results_root / '59_blueprint_ambiguity_resolution_actions.tsv'
summary_review_path = results_root / '60_blueprint_ambiguity_resolution_summary.tsv'

inventory_sorted_df.to_csv(inventory_review_path, sep='\t', index=False)
blocking_review_df.to_csv(blocking_review_path, sep='\t', index=False)
provisional_review_df.to_csv(provisional_review_path, sep='\t', index=False)
manual_review_candidates_df.to_csv(manual_review_path, sep='\t', index=False)
later_xml_review_df.to_csv(later_xml_review_path, sep='\t', index=False)
actions_review_df.to_csv(actions_review_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f'Saved: {inventory_review_path}')
print(f'Saved: {blocking_review_path}')
print(f'Saved: {provisional_review_path}')
print(f'Saved: {manual_review_path}')
print(f'Saved: {later_xml_review_path}')
print(f'Saved: {actions_review_path}')
print(f'Saved: {summary_review_path}')

display(actions_review_df)
display(summary_review_df)


This notebook remains review-only. It does not parse raw files, construct a cohort table, or freeze an endpoint.
